# Psychiatric Comorbidities in Epilepsy: Surgical vs. Non-Surgical
## Cohort Identification & Prevalence Analysis

**Data source:** MIMIC-IV v3.1  
**Objective:** Identify epilepsy patients, flag surgical cases, and compute psychiatric comorbidity prevalence.

In [ ]:
import duckdb
import pandas as pd
import sys
sys.path.insert(0, '.')
from icd_codes import (
    EPILEPSY_ICD10, EPILEPSY_ICD9,
    EPILEPSY_SURGERY_ICD10_PCS, EPILEPSY_SURGERY_ICD9,
    PSYCH_CATEGORIES, build_icd_prefix_sql, get_all_psych_codes
)

DB_PATH = "/Volumes/Niels 2/MIMIC/databases/mimic4.db"
con = duckdb.connect(DB_PATH, read_only=True)
print(f"Connected to MIMIC-IV at {DB_PATH}")

## 1. How many epilepsy patients exist in MIMIC-IV?

In [ ]:
# Count unique patients and admissions with any epilepsy diagnosis
epilepsy_counts = con.sql("""
    SELECT 
        COUNT(DISTINCT subject_id) AS unique_patients,
        COUNT(DISTINCT hadm_id) AS unique_admissions
    FROM mimiciv_hosp.diagnoses_icd
    WHERE (icd_version = 10 AND icd_code LIKE 'G40%')
       OR (icd_version = 9 AND icd_code LIKE '345%')
""").df()

print("=== Epilepsy Patients in MIMIC-IV ===")
print(f"Unique patients: {epilepsy_counts['unique_patients'].iloc[0]:,}")
print(f"Unique admissions: {epilepsy_counts['unique_admissions'].iloc[0]:,}")

In [ ]:
# Breakdown by ICD version and intractable vs not
con.sql("""
    SELECT 
        icd_version,
        icd_code,
        COUNT(DISTINCT subject_id) AS patients,
        COUNT(DISTINCT hadm_id) AS admissions
    FROM mimiciv_hosp.diagnoses_icd
    WHERE (icd_version = 10 AND icd_code LIKE 'G40%')
       OR (icd_version = 9 AND icd_code LIKE '345%')
    GROUP BY icd_version, icd_code
    ORDER BY patients DESC
""").df()

## 2. Identify Epilepsy Surgery Patients

In [ ]:
# Build procedure code list for SQL
icd10_proc_codes = list(EPILEPSY_SURGERY_ICD10_PCS.keys())
icd9_proc_codes = list(EPILEPSY_SURGERY_ICD9.keys())

icd10_proc_str = ", ".join(f"'{c}'" for c in icd10_proc_codes)
icd9_proc_str = ", ".join(f"'{c}'" for c in icd9_proc_codes)

# Find patients with epilepsy who also had epilepsy surgery procedures
surgery_query = f"""
    WITH epilepsy_patients AS (
        SELECT DISTINCT subject_id, hadm_id
        FROM mimiciv_hosp.diagnoses_icd
        WHERE (icd_version = 10 AND icd_code LIKE 'G40%')
           OR (icd_version = 9 AND icd_code LIKE '345%')
    ),
    surgery_patients AS (
        SELECT DISTINCT subject_id, hadm_id
        FROM mimiciv_hosp.procedures_icd
        WHERE (icd_version = 10 AND icd_code IN ({icd10_proc_str}))
           OR (icd_version = 9 AND icd_code IN ({icd9_proc_str}))
    )
    SELECT 
        'Epilepsy + Surgery' AS cohort,
        COUNT(DISTINCT e.subject_id) AS patients,
        COUNT(DISTINCT e.hadm_id) AS admissions
    FROM epilepsy_patients e
    INNER JOIN surgery_patients s 
        ON e.subject_id = s.subject_id AND e.hadm_id = s.hadm_id
"""

surgery_counts = con.sql(surgery_query).df()
print("=== Epilepsy Surgery Patients ===")
print(surgery_counts)

In [ ]:
# What specific procedures are found?
proc_detail_query = f"""
    SELECT 
        p.icd_version,
        p.icd_code,
        COALESCE(d.long_title, 'Unknown') AS description,
        COUNT(DISTINCT p.subject_id) AS patients
    FROM mimiciv_hosp.procedures_icd p
    LEFT JOIN mimiciv_hosp.d_icd_procedures d 
        ON p.icd_code = d.icd_code AND p.icd_version = d.icd_version
    WHERE p.subject_id IN (
        SELECT DISTINCT subject_id
        FROM mimiciv_hosp.diagnoses_icd
        WHERE (icd_version = 10 AND icd_code LIKE 'G40%')
           OR (icd_version = 9 AND icd_code LIKE '345%')
    )
    AND (
        (p.icd_version = 10 AND p.icd_code IN ({icd10_proc_str}))
        OR (p.icd_version = 9 AND p.icd_code IN ({icd9_proc_str}))
    )
    GROUP BY p.icd_version, p.icd_code, d.long_title
    ORDER BY patients DESC
"""

con.sql(proc_detail_query).df()

## 3. Build the Full Cohort Table

In [ ]:
# Create the main cohort: epilepsy patients with demographics and surgery flag
cohort_query = f"""
    WITH epilepsy_admissions AS (
        SELECT DISTINCT subject_id, hadm_id
        FROM mimiciv_hosp.diagnoses_icd
        WHERE (icd_version = 10 AND icd_code LIKE 'G40%')
           OR (icd_version = 9 AND icd_code LIKE '345%')
    ),
    surgery_admissions AS (
        SELECT DISTINCT subject_id, hadm_id
        FROM mimiciv_hosp.procedures_icd
        WHERE (icd_version = 10 AND icd_code IN ({icd10_proc_str}))
           OR (icd_version = 9 AND icd_code IN ({icd9_proc_str}))
    )
    SELECT 
        e.subject_id,
        e.hadm_id,
        p.gender,
        p.anchor_age,
        a.race,
        a.insurance,
        a.marital_status,
        a.admission_type,
        a.admittime,
        a.dischtime,
        a.hospital_expire_flag,
        CASE WHEN s.hadm_id IS NOT NULL THEN 1 ELSE 0 END AS had_epilepsy_surgery,
        DATE_DIFF('day', a.admittime, a.dischtime) AS los_days
    FROM epilepsy_admissions e
    JOIN mimiciv_hosp.patients p ON e.subject_id = p.subject_id
    JOIN mimiciv_hosp.admissions a ON e.hadm_id = a.hadm_id
    LEFT JOIN surgery_admissions s 
        ON e.subject_id = s.subject_id AND e.hadm_id = s.hadm_id
"""

cohort = con.sql(cohort_query).df()
print(f"Total epilepsy admissions: {len(cohort):,}")
print(f"  Surgical: {cohort['had_epilepsy_surgery'].sum():,}")
print(f"  Non-surgical: {(cohort['had_epilepsy_surgery'] == 0).sum():,}")
print(f"\nUnique patients: {cohort['subject_id'].nunique():,}")

In [ ]:
# Demographics overview
print("=== Demographics ===")
print("\nGender:")
print(cohort.groupby('had_epilepsy_surgery')['gender'].value_counts(normalize=True).round(3))
print("\nAge (mean +/- SD):")
print(cohort.groupby('had_epilepsy_surgery')['anchor_age'].agg(['mean', 'std']).round(1))
print("\nRace:")
print(cohort.groupby('had_epilepsy_surgery')['race'].value_counts(normalize=True).round(3))

## 4. Psychiatric Comorbidity Prevalence

In [ ]:
# For each psychiatric category, compute prevalence in surgical vs non-surgical
results = []

for cat_name, cat_data in PSYCH_CATEGORIES.items():
    # Build SQL for this category
    icd10_conditions = " OR ".join(f"icd_code LIKE '{c}%'" for c in cat_data['icd10'])
    icd9_conditions = " OR ".join(f"icd_code LIKE '{c}%'" for c in cat_data['icd9'])
    
    psych_query = f"""
        SELECT DISTINCT subject_id, hadm_id
        FROM mimiciv_hosp.diagnoses_icd
        WHERE (icd_version = 10 AND ({icd10_conditions}))
           OR (icd_version = 9 AND ({icd9_conditions}))
    """
    
    psych_df = con.sql(psych_query).df()
    psych_hadm_ids = set(zip(psych_df['subject_id'], psych_df['hadm_id']))
    
    # Flag each admission
    cohort[f'has_{cat_name}'] = cohort.apply(
        lambda row: 1 if (row['subject_id'], row['hadm_id']) in psych_hadm_ids else 0, 
        axis=1
    )
    
    # Compute prevalence
    for surgery_flag in [0, 1]:
        subset = cohort[cohort['had_epilepsy_surgery'] == surgery_flag]
        n = len(subset)
        n_with = subset[f'has_{cat_name}'].sum()
        prev = n_with / n * 100 if n > 0 else 0
        results.append({
            'category': cat_data['label'],
            'group': 'Surgical' if surgery_flag == 1 else 'Non-Surgical',
            'n_with_dx': int(n_with),
            'n_total': n,
            'prevalence_pct': round(prev, 1),
        })

prevalence_df = pd.DataFrame(results)
print("=== Psychiatric Comorbidity Prevalence ===")
prevalence_df

In [ ]:
# Pivot for a cleaner comparison table
pivot = prevalence_df.pivot_table(
    index='category', 
    columns='group', 
    values='prevalence_pct'
).round(1)

# Add counts
counts_pivot = prevalence_df.pivot_table(
    index='category', 
    columns='group', 
    values='n_with_dx'
)

# Combine
comparison = pd.DataFrame()
for group in ['Non-Surgical', 'Surgical']:
    if group in pivot.columns:
        comparison[f'{group} n'] = counts_pivot[group].astype(int)
        comparison[f'{group} %'] = pivot[group]

comparison = comparison.sort_values(by=comparison.columns[1], ascending=False)
print("\n=== Comparison: Surgical vs Non-Surgical ===")
comparison

In [ ]:
# Any psychiatric comorbidity (excluding PNES)
psych_cols = [c for c in cohort.columns if c.startswith('has_') and c != 'has_pnes']
cohort['any_psych'] = (cohort[psych_cols].sum(axis=1) > 0).astype(int)

for surgery_flag, label in [(0, 'Non-Surgical'), (1, 'Surgical')]:
    subset = cohort[cohort['had_epilepsy_surgery'] == surgery_flag]
    n = len(subset)
    n_any = subset['any_psych'].sum()
    pct = n_any / n * 100 if n > 0 else 0
    print(f"{label}: {n_any}/{n} ({pct:.1f}%) have any psychiatric comorbidity")

## 5. Save Cohort for Further Analysis

In [ ]:
# Save the cohort with all flags
output_path = '/Volumes/Niels 2/MIMIC/physionet.org/files/mimiciv/3.1/analysis/epilepsy_psych/epilepsy_cohort.csv'
cohort.to_csv(output_path, index=False)
print(f"Cohort saved to {output_path}")
print(f"Shape: {cohort.shape}")
print(f"Columns: {list(cohort.columns)}")

In [ ]:
con.close()
print("Done!")